# MirrorTopology Step 1 — Phase D-2：保存 bank の実行後検証（read-only；family 単位；v0.2）
Drive 上の 1 family の出力（`<RUN>/out`）を，受入れ済み commit の engine で**読取り専用**に検証する：全 directory の `verify_bank_dir`（bytes・sidecar⇄manifest・member 集合・dtype・有限性・cid・範囲・UID・key），NPZ file SHA の再計算と registry／final record の照合，role×selection ごとの配列統計，必須 f32 subset（batch 0）と W₂ bank の paired f64/f32 感度（axis／plane flip 率，flip 行の |ΔT|，near-tie 数），同 latent 不変量。Drive は変更しない。生成・較正・判定は行わない。**v0.2**：受入れ済み run への束縛（ledger の固定 record／unit SHA と正規 spec の必要 unit 集合），検証 source の inventory 束縛（生成 source と別記録），出力先の隔離（run／bank と交差する OUT・既存 alias は書込み前に拒否），登録 f32 規則（選択 S⁺ の相対差 <1e-6 の near-tie・Event B mismatch ≤1e-5・flip evidence）。`all_ok` は file/member integrity，`f32_registered_gate` は候補評価であり，いずれも科学的 support/strong ではない。出力は小さな JSON（`d2_verify_<FAMILY>.json`）と zip。

In [ ]:
# --- 0. OUTER LOCK (the only editable cell)
REPO_URL = 'https://github.com/tsujikeita/mirror-topology.git'
REPO_COMMIT = '<full 40-hex commit of the verification target (must contain d/d2_verify_banks.py)>'
EXPECTED_INVENTORY_SHA256 = '<sha256 of engine/phaseB/B2_completion_inventory.json inside that commit>'
FAMILY = 'E1'
RUN_ROOT = '/content/drive/MyDrive/MirrorTopology_D2/E1_20260923T063210Z/out'    # the family run's out/ (contains d2/ and d2_final_record.json)


In [ ]:
# --- 1. fresh scratch checkout; inventory + script bound; Drive mounted (read-only usage)
import subprocess, sys, os, json, hashlib, shutil, time, re
sha=lambda p: hashlib.sha256(open(p,'rb').read()).hexdigest()
assert re.fullmatch(r'[0-9a-f]{40}', REPO_COMMIT) and re.fullmatch(r'[0-9a-f]{64}', EXPECTED_INVENTORY_SHA256) and FAMILY in ('E1','E2','E7','E8')
from google.colab import drive; drive.mount('/content/drive'); assert os.path.exists(f'{RUN_ROOT}/d2/d2_bank_registry.json'), RUN_ROOT
SCRATCH='/content/d2v_scratch'; shutil.rmtree(SCRATCH, ignore_errors=True); subprocess.run(['git','clone','-q',REPO_URL,SCRATCH],check=True); subprocess.run(['git','-C',SCRATCH,'checkout','-q',REPO_COMMIT],check=True)
assert subprocess.check_output(['git','-C',SCRATCH,'rev-parse','HEAD']).decode().strip()==REPO_COMMIT and subprocess.check_output(['git','-C',SCRATCH,'status','--porcelain']).decode().strip()==''
PHASEB=f'{SCRATCH}/engine/phaseB'; INV=f'{PHASEB}/B2_completion_inventory.json'; assert sha(INV)==EXPECTED_INVENTORY_SHA256; inv=json.load(open(INV)); SCRIPT=f'{PHASEB}/d/d2_verify_banks.py'; assert sha(SCRIPT)==inv['d_sha256']['d/d2_verify_banks.py']
pins=json.load(open(f'{PHASEB}/d/d2_pins.json')); ex=pins['environment']; subprocess.run([sys.executable,'-m','pip','install','-q',f"numpy=={ex['numpy']}",f"scipy=={ex['scipy']}",f"healpy=={ex['healpy']}",f"pot=={ex['pot']}",'threadpoolctl'],check=True)
OUT=f'/content/d2_verify_{FAMILY}'; shutil.rmtree(OUT, ignore_errors=True); os.makedirs(OUT); lock=dict(commit=REPO_COMMIT, inventory_sha256=sha(INV), script_sha256=sha(SCRIPT), family=FAMILY, run_root=RUN_ROOT, engine=inv['engine_version']); json.dump(lock, open(f'{OUT}/verify_lock.json','w'), indent=1); print(lock)


In [ ]:
# --- 2. read-only verification (reads every NPZ once; ~10-20 min per family)
assert not os.path.exists(f'{OUT}/d2_verify_{FAMILY}.json'); rc=subprocess.run([sys.executable,SCRIPT,'--phaseb',PHASEB,'--run-root',RUN_ROOT,'--out',f'{OUT}/report','--mode','accepted'],capture_output=True,text=True)
# accepted mode; preserve stdout/stderr before opening the report
open(f'{OUT}/verify_stdout.txt','w').write(rc.stdout); open(f'{OUT}/verify_stderr.txt','w').write(rc.stderr); print(rc.stdout[-2500:]); print('rc', rc.returncode)
vp=f'{OUT}/report/d2_verify_{FAMILY}.json'
v=json.load(open(vp)) if os.path.isfile(vp) else dict(stage='report_missing', all_ok=False, coverage_complete=False, accepted_run_binding=False, failures=['verifier did not write its report; see stdout/stderr'])
print('rc', rc.returncode, '| accepted_run_binding', v.get('accepted_run_binding'), '| scope', v.get('verification_scope'), '| all_ok', v.get('all_ok'), '| coverage_complete', v.get('coverage_complete'), '| f32 gate', (v.get('f32_registered_gate') or {}).get('status'), (v.get('f32_registered_gate') or {}).get('all_pairs_pass_registered_rule'), '| failures', v.get('failures'), '| seconds', round(v.get('seconds', 0)))
json.dump(dict(lock=lock, exit_code=rc.returncode, stage=v.get('stage'), all_ok=v.get('all_ok'), coverage_complete=v.get('coverage_complete'), accepted_run_binding=v.get('accepted_run_binding')), open(f'{OUT}/verify_final_record.json','w'), indent=1)


In [ ]:
# --- 3. zip the evidence (small) for the audit
zp=f'/content/d2_verify_{FAMILY}_{REPO_COMMIT[:12]}.zip'; shutil.make_archive(zp[:-4], 'zip', root_dir=OUT)
from google.colab import files; print(zp, os.path.getsize(zp)); files.download(zp)
